# Stage C 03d — paper-deep memory T4 stability

Run after 03c. This notebook runs small CUDA correctness smokes, then separate 500-step compact soaks for the exact paper recurrence and the preregistered RMS-stabilized recurrence. Each run is checkpointed, logged to Drive, and deterministically reloaded/continued twice.

In [ ]:
# USER CONFIGURATION
REPO_URL='https://github.com/Gonza10V/SeqTrainer.git'
GIT_REF='2cf1e9fc87025a71b10121ba9452c8175ec0fbb3'
DRIVE_ROOT='/content/drive/MyDrive/SeqTrainerStageC'
BLOCK_COUNT=4; D_MODEL=128; NUM_HEADS=4; HORIZON=3
SOAK_STEPS=500; CHECKPOINT_EVERY=100; VALIDATION_STREAMS=4
POLICIES=('paper_exact','stabilized_rms_v1')


In [ ]:
from pathlib import Path
from google.colab import drive
import json, subprocess, sys
mount=Path('/content/drive')
if not (mount/'MyDrive').is_dir(): drive.mount(str(mount),timeout_ms=120000)
repo=Path('/content/SeqTrainer')
if not repo.exists(): subprocess.run(['git','clone',REPO_URL,str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin'],check=True)
subprocess.run(['git','-C',str(repo),'checkout',GIT_REF],check=True)
subprocess.run([sys.executable,'-m','pip','install','-e',f'{repo}[torch,bacteria-titan]'],check=True)
import torch
device=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no CUDA device'
if 'T4' not in device.upper(): raise RuntimeError(f'Select a T4 runtime; assigned: {device}')
selection=json.loads((Path(DRIVE_ROOT)/'runs/c1_tokenizers_cpu/tokenizer_selection.json').read_text())
dataset=Path(DRIVE_ROOT)/'stage_c_dataset/ordered_streams'/selection['selected_tokenizer']
if not (dataset/'token_stream_manifest.json').is_file(): raise FileNotFoundError('Run 00b first')
PROTOCOL=repo/'studies/stage_c_ecoli_escherichia_paper_deep_memory_v2/protocol.json'
STUDY_ROOT=Path(DRIVE_ROOT)/'study/stage_c_ecoli_escherichia_paper_deep_memory_v2'
subprocess.run(['seqtrainer-titans-stage-c-study','initialize','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT)],check=True)
print('Dataset:',dataset); print('Device:',device)


In [ ]:
def run_logged(run_dir,label,command):
    try:
        subprocess.run(['seqtrainer-titans-stage-c-colab-run','--run-dir',str(run_dir),'--label',label,'--repo',str(repo),'--',*command],check=True)
    except subprocess.CalledProcessError:
        failure=run_dir/'FAILED.txt'; log=run_dir/'logs'/f'{label}.log'
        if failure.exists(): print(failure.read_text(errors='replace'))
        if log.exists(): print(log.read_text(errors='replace')[-16000:])
        raise

# Small geometry keeps CPU/GPU parity practical while exercising the full deep path.
smoke_root=Path(DRIVE_ROOT)/'runs/c13_paper_deep_t4_smoke'
for policy in POLICIES:
    out=smoke_root/f'{policy}.json'
    run_logged(smoke_root, f'smoke_{policy}', ['seqtrainer-titans-stage-c-gpu-smoke','--dataset-dir',str(dataset),'--output',str(out),'--require','T4','--block-count','1','--d-model','32','--num-heads','4','--persistent-tokens','4','--memory-depth','2','--gradient-horizon','1','--memory-architecture','paper_residual_mlp_v2','--memory-recurrence-policy',policy])
print('Both CUDA smokes passed.')


In [ ]:
def deep_flags(policy):
    flags=['--memory-architecture','paper_residual_mlp_v2','--memory-depth','2','--memory-expansion-factor','4','--memory-projection-convolution-kernel','4','--memory-normalize-queries-and-keys','--memory-gate-granularity','per_layer_channel','--memory-recurrence-policy',policy,'--memory-surprise-clip-norm','none','--memory-alpha-initial','0.001','--memory-eta-initial','0.9','--memory-theta-initial','0.001']
    if policy=='paper_exact': return flags+['--memory-associative-loss-reduction','sum','--memory-max-gradient-rms','none','--memory-max-gradient-rms-ratio','none','--memory-theta-max','1.0']
    return flags+['--memory-associative-loss-reduction','mean','--memory-max-gradient-rms','none','--memory-max-gradient-rms-ratio','10.0','--memory-theta-max','0.5']

soak_root=Path(DRIVE_ROOT)/'runs/c14_paper_deep_t4_500step_soaks'
for policy in POLICIES:
    run_id=f'deep_t4_soak_{"exact" if policy=="paper_exact" else "stabilized"}_500'
    run_dir=soak_root/policy
    command=['seqtrainer-titans-stage-c-train','--dataset-dir',str(dataset),'--run-dir',str(run_dir),'--memory-mode','adaptive','--horizon',str(HORIZON),'--batch-size','1','--max-optimizer-steps',str(SOAK_STEPS),'--checkpoint-every',str(CHECKPOINT_EVERY),'--learning-rate','3e-5','--gradient-clip-norm','0.5','--validation-streams',str(VALIDATION_STREAMS),'--activation','float32','--block-count',str(BLOCK_COUNT),'--d-model',str(D_MODEL),'--num-heads',str(NUM_HEADS),'--persistent-tokens','4',*deep_flags(policy),'--protocol',str(PROTOCOL),'--run-id',run_id]
    run_logged(run_dir,f'train_{policy}',command)
    resume_out=run_dir/'resume_verification.json'
    run_logged(run_dir,f'resume_{policy}',['seqtrainer-titans-stage-c-resume-verify','--dataset-dir',str(dataset),'--checkpoint',str(run_dir/'latest.pt'),'--output',str(resume_out),'--device','cuda','--expected-step',str(SOAK_STEPS)])
    subprocess.run(['seqtrainer-titans-stage-c-study','record','--protocol',str(PROTOCOL),'--study-root',str(STUDY_ROOT),'--run-id',run_id,'--evidence-tier','confirmatory','--artifact',str(run_dir)],check=True)
print('SHARE THIS DIRECTORY:',soak_root)
print('Review validation.json, training_history.json, memory_conditioning.svg, and resume_verification.json for both policies before 03e.')
